# New York Times Article Analysis

This project uses the NYT Article Search API to collect article data, including titles and thumbnail images, which are then analyzed using machine learning.

The goal is to examine the distribution of article sentiment (negative, neutral, positive) and to measure the representation of men vs. women in article thumbnail images.

Importing dependancies that are required for the project, as well as turning off warning reporting to avoid clutter during image captioning.

In [ ]:
import requests
import time
from dotenv import load_dotenv
import os
from io import BytesIO
from transformers import logging
from tqdm import tqdm
import pandas as pd
from IPython.display import display
logging.set_verbosity_error()

Loading API key required for API calls from .env file.

In [ ]:
load_dotenv()
API_KEY = os.getenv("NYT_API_KEY")

Defining a function that calls on the NYT Article Search API. Each page has at most 10 articles, so this function by default attemps to read 10 pages for a total of 100 articles. Using time.sleep at the end of the function to avoid rate limiting from the API.

In [ ]:
def get_search_results(query="finance", max_pages=10):
    articles = []

    for page in range(max_pages):
        url = "https://api.nytimes.com/svc/search/v2/articlesearch.json"
        params = {
            "q": query,
            "api-key": API_KEY,
            "page": page
        }
        response = requests.get(url, params=params)
        data = response.json()

        hits = data.get("response", {}).get("docs", [])
        if not hits:
            break  # No more results

        articles.extend(hits)
        print(f"Page {page}: fetched {len(hits)} articles (total: {len(articles)})")

        time.sleep(12)  # NYT rate limit: 5 requests/min → 1 req per 12s

    return articles


In [ ]:
data = get_search_results()

Creating a list of dictionaries that include each article's title and thumbnail. Gets lower resolution thumbnail if higher resolution version was not found.

In [ ]:
results = []
for story in data:
  result = {}
  result["title"] = story.get("headline", {}).get("main")
  multimedia = story["multimedia"]
  if multimedia["default"].get("url"):
    result["image_url"] = multimedia["default"]["url"]
  elif multimedia["thumbnail"] and multimedia["thumbnail"].get("url"):
    result["image_url"] = multimedia["thumbnail"]["url"]
  results.append(result)

Loading sentiment analysis pipeline. This project uses a RoBERTa based sentiment analysis model trained on Twitter posts.

In [ ]:
from transformers import pipeline

sentiment_pipeline = pipeline('sentiment-analysis', model='cardiffnlp/twitter-roberta-base-sentiment-latest')

print("Sentiment analysis model loaded successfully.")

Using BLIM image captioning pipeline to gather captions on thumbnails.

In [ ]:
from PIL import Image

image_caption_pipeline = pipeline(
    "image-text-to-text",
    model="Salesforce/blip-image-captioning-base"
)


Applying sentiment analysis pipeline on our article titles.

In [ ]:
articles_with_sentiment = []

for article in results:
    title = article.get('title')
    if title:
        title = str(title)
        sentiment_result = sentiment_pipeline(title)[0]
        article_with_sentiment = article.copy()
        article_with_sentiment['sentiment_label'] = sentiment_result['label']
        article_with_sentiment['sentiment_score'] = sentiment_result['score']
        articles_with_sentiment.append(article_with_sentiment)
    else:
        articles_with_sentiment.append(article.copy())

print(f"Processed {len(articles_with_sentiment)} articles with sentiment.")
print("First 5 articles with sentiment scores:")
for i in range(min(5, len(articles_with_sentiment))):
    print(articles_with_sentiment[i])

Applying image captioning pipeline on our article thumbnails.

In [ ]:
for article in tqdm(articles_with_sentiment):
    img_url = article.get("image_url")
    if img_url:
      response = requests.get(img_url)
      try:
          image = Image.open(BytesIO(response.content)).convert("RGB")
          caption = image_caption_pipeline(image, text="", max_new_tokens = 50)[0]['generated_text']
          article['image_caption'] = caption
      except Exception as e:
          article['image_caption'] = None
          print(f"Failed to caption {img_url}: {e}")
    else:
        article['image_caption'] = None

print("First 5 articles with sentiment and image captions:")
for i in range(min(5, len(articles_with_sentiment))):
    print(articles_with_sentiment[i])


Summary of gathered data and machine learning analysis results:

In [ ]:

df_articles = pd.DataFrame(articles_with_sentiment)
print("First 5 rows of the DataFrame:")
display(df_articles)

#Results
Calculating distribution of article sentiments:

In [ ]:
counts = {"negative" : 0, "neutral": 0, "positive": 0}
for article in articles_with_sentiment:
  sentiment_label = article.get("sentiment_label")
  if sentiment_label:
    counts[sentiment_label] += 1
print(f"{round(counts["negative"]/len(articles_with_sentiment)*100,2)}% of articles are negative")
print(f"{round(counts["neutral"]/len(articles_with_sentiment)*100,2)}% of articles are neutral")
print(f"{round(counts["positive"]/len(articles_with_sentiment)*100,2)}% of articles are positive")
print(f"Sample size: {len(articles_with_sentiment)}")

Calculating representation of men vs. women in article thumbnail images:

In [ ]:
keywords = {"Man": ["man", "men", "boy", "boys"],
            "Woman": ["woman", "women", "girl", "girls"]}
counts =  {label: 0 for label in keywords}

for article in articles_with_sentiment:
    caption = article.get("image_caption")
    if caption:
      for gender in keywords.keys():
        if any(word in caption.lower() for word in keywords[gender]):
          counts[gender] += 1
print(f"Man/Men found in {counts["Man"]} article thumbnails, or {counts["Man"]/len(articles_with_sentiment)*100}% of article thumbnails")
print(f"Woman/Women found in {counts["Woman"]} article thumbnails, or {counts["Woman"]/len(articles_with_sentiment)*100}% of article thumbnails")